# Scarlet collaborator reproduction

This notebook is a thin interface to the tested benchmark commands. It runs one predeclared start, plots Scarlet's own saved product, and reports both truth-independent fit diagnostics and truth-referenced recovery diagnostics. Do not choose a start using the latter; run A, B, and C as a declared set.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

# Configuration: edit only this cell. Run the notebook from the Scarlet checkout.
SCARLET_REPO = Path.cwd().resolve()
DATA_ROOT = Path('/path/to/collaborator/data')
OUTPUT_ROOT = SCARLET_REPO / 'benchmark_artifacts' / 'collaborator_reproduction'
START = 'A'  # Repeat for A, B, and C; never select using truth.
MAX_ITER = 300
FIT_DTYPE = 'float32'
CHANNEL_CHUNK_SIZE = 64

In [ ]:
run_dir = OUTPUT_ROOT / f'start{START}'
fit_command = [
    sys.executable, '-m', 'benchmarks.run_collaborator_reproduction',
    '--data-root', str(DATA_ROOT),
    '--output-dir', str(run_dir),
    '--start', START,
    '--max-iter', str(MAX_ITER),
    '--dtype', FIT_DTYPE,
    '--channel-chunk-size', str(CHANNEL_CHUNK_SIZE),
    '--optimality-tolerance', '1e-4',
]
fit_command

The next cell performs the fit and writes the NPZ and JSON provenance products. For a full A/B/C campaign on a cluster, use `submit_collaborator_reproduction.sh` instead.

In [ ]:
subprocess.run(fit_command, cwd=SCARLET_REPO, check=True)

In [ ]:
product = run_dir / f'scarlet_matched_start{START}.npz'
metrics = run_dir / 'scarlet_matched_metrics.json'
plot_dir = run_dir / 'plots'
plot_command = [
    sys.executable, '-m', 'benchmarks.plot_collaborator_reproduction',
    '--product', str(product),
    '--metrics', str(metrics),
    '--output-dir', str(plot_dir),
]
subprocess.run(plot_command, cwd=SCARLET_REPO, check=True)

In [ ]:
report = json.loads(metrics.read_text())
{
    'start': report['start'],
    'iterations': report['iterations'],
    'runtime_seconds': report['runtime_seconds'],
    'optimality_converged': report['optimality_converged'],
    'projected_gradient': report['parameter_relative_projected_gradient'],
    'residual': report['residual'],
    'ifu_ingestion': report['ifu_ingestion'],
    'structural_mixing': report['structural_mixing'],
}

In [ ]:
from IPython.display import Image, display

for name in (
    'scarlet_collaborator_spectra.png',
    'scarlet_collaborator_morphologies.png',
    'scarlet_collaborator_residual.png',
):
    display(Image(filename=str(plot_dir / name)))